# ch02 练习：特征工程

与讲解版逐 Cell 对应。核心代码被挖空，`assert` 验收**保留**——
跑通所有 `assert` 即自查通过。卡住就回讲解版看对应小节。

## 一、考点回顾

| 竞赛评分点 | 本节覆盖 |
|---|---|
| 模型训练 40% | 三种 scaler / SimpleImputer / 两种 encoder / PolynomialFeatures |
| 模型选择 10% | 预处理方式与模型类型匹配 |

## 二、API 速查

| 类 | 关键参数 | 一句话 |
|---|---|---|
| `StandardScaler` | — | 减均值除标准差 |
| `MinMaxScaler` | — | 压 [0,1]，测试段可越界 |
| `RobustScaler` | — | 中位数 + IQR |
| `SimpleImputer` | `strategy` `add_indicator` | 缺失填充 |
| `OneHotEncoder` | `drop` `min_frequency` `handle_unknown` | 类别 → 独热 |
| `OrdinalEncoder` | `unknown_value` | 类别 → 整数 |
| `PolynomialFeatures` | `degree` `include_bias` `interaction_only` | 升维（拒绝 NaN） |

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.simplefilter("ignore")
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

import sklearn
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import (MinMaxScaler, OneHotEncoder, OrdinalEncoder,
                                   PolynomialFeatures, RobustScaler,
                                   StandardScaler)

DATA = Path("data")

dev = pd.read_csv(DATA / "defect_ml.csv")   # 二分类：预测「是否危急」

NUM = ["负荷值", "温度", "湿度", "投运年限", "巡检耗时分钟"]
CAT = ["设备类型", "缺陷类型"]
TARGET = "是否危急"

X = dev[NUM + CAT]
y = dev[TARGET]

# 分层切分（ch01 的结论：分类任务 stratify=y 无条件写上）
Xs_tr, Xs_te, ys_tr, ys_te = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print("sklearn", sklearn.__version__)
print("train", Xs_tr.shape, "test", Xs_te.shape)
print("训练段数值列缺失:", Xs_tr[NUM].isna().sum().tolist())

In [ ]:
# 脚手架：写在 @@todo 之外，练习版原样保留
def probe(fn, *args, **kwargs):
    """把「调用可能抛异常」写成返回值，避免在挖空块里写 try/except。

    返回 ("ok", 结果) 或 ("err", 异常类型名)。
"""
    try:
        return "ok", fn(*args, **kwargs)
    except Exception as exc:
        return "err", type(exc).__name__


print("脚手架就绪：probe")

## 题 1：scaler 三兄弟同台（对应讲解 §3.1）

In [ ]:
# TODO(1)：在训练段数值列上 fit StandardScaler / MinMaxScaler / RobustScaler
# 提示：StandardScaler().fit(Xs_tr[NUM])，三兄弟同一份数据
ss = ____
mm = ____
rs = ____

print("StandardScaler mean_:", np.round(ss.mean_, 4).tolist())
print("StandardScaler scale_:", np.round(ss.scale_, 4).tolist())
print("MinMaxScaler  data_min_:", mm.data_min_.tolist())
print("MinMaxScaler  data_max_:", mm.data_max_.tolist())
print("RobustScaler  center_ :", np.round(rs.center_, 4).tolist())
print("RobustScaler  scale_ :", np.round(rs.scale_, 4).tolist())

# RobustScaler 的 center_ 就是中位数，scale_ 就是 IQR —— 和手算对账
q1 = Xs_tr["负荷值"].quantile(0.25)
q3 = Xs_tr["负荷值"].quantile(0.75)
print("负荷值 IQR 手算:", round(float(q3 - q1), 4), "vs RobustScaler scale_[0]:",
      round(float(rs.scale_[0]), 4))
print("RobustScaler center_ == SimpleImputer(median) 的统计量:",
      bool(np.allclose(rs.center_, [427.33, 35.1, 64.9, 16.0, 44.0])))

assert round(float(ss.mean_[0]), 4) == 422.6926, "训练段负荷值均值"
assert round(float(ss.scale_[0]), 4) == 154.4385, "训练段负荷值标准差"
assert float(mm.data_min_[0]) == 40.0 and float(mm.data_max_[0]) == 835.12
assert round(float(rs.center_[0]), 4) == 427.33 and round(float(rs.scale_[0]), 4) == 215.18
assert round(float(q3 - q1), 4) == 215.18

**为什么这样做**：三个缩放器记的统计量完全不同——均值/标准差、
min/max、中位数/IQR。RobustScaler 的 center_ 和 scale_ 必须能与
pandas 手算的中位数、IQR 对上账，这一致性就是你的验收标准。

## 题 2：NaN 穿透与测试段行为（对应讲解 §3.2-3.3）

In [ ]:
# TODO(2)：用训练段 fit 好的 ss / mm 分别 transform 训练段与测试段数值列
# 提示：ss.transform(Xs_tr[NUM])；测试段只许 transform 不许重新 fit
tr_t = ____
te_t = ____

# TODO(3)：再对测试段做 MinMaxScaler 变换，数一数越出 [0, 1] 的元素个数
# 提示：mm.transform(Xs_te[NUM])；(te_mt > 1).sum() 与 (te_mt < 0).sum()
te_mt = ____
n_over_hi = ____
n_over_lo = ____

print("训练段 transform 后 nanmean:", np.round(np.nanmean(tr_t, axis=0), 10).tolist()[:3], "...")
print("训练段 transform 后 nanstd :", np.round(np.nanstd(tr_t, axis=0), 4).tolist())
print("transform 输出里的 NaN 个数:", int(np.isnan(tr_t).sum()),
      "← 负荷值 36 + 温度 36，原样穿透")
print("测试段 transform 后 nanmean:", np.round(np.nanmean(te_t, axis=0), 4).tolist())
print("测试段 transform 后 nanstd :", np.round(np.nanstd(te_t, axis=0), 4).tolist())
print("MinMax 测试段越界: >1 有", n_over_hi, "个 | <0 有", n_over_lo, "个")

assert np.allclose(np.nanmean(tr_t, axis=0), 0, atol=1e-10), "训练段均值恰为 0"
assert np.allclose(np.nanstd(tr_t, axis=0), 1, atol=1e-10), "训练段标准差恰为 1"
assert int(np.isnan(tr_t).sum()) == 72, "NaN 原样穿透：36 + 36"
assert round(float(np.nanmean(te_t, axis=0)[0]), 4) == 0.0832, "测试段均值不在 0"
assert round(float(np.nanstd(te_t, axis=0)[0]), 4) == 0.9115, "测试段标准差不在 1"
assert n_over_hi == 3, "温度 1 个 + 湿度 2 个"
assert n_over_lo == 0
assert round(float(np.nanmax(te_mt[:, 1])), 4) == 1.0045, "温度越界幅度不大但确实越了"
assert round(float(np.nanmax(te_mt[:, 2])), 4) == 1.0180
assert round(float(np.nanmax(te_mt[:, 0])), 4) == 0.9556, "负荷值 max 恰好落在训练段内"

## 题 3：SimpleImputer 三种策略（对应讲解 §3.4）

In [ ]:
# TODO(4)：定义 median 策略 + add_indicator 的 SimpleImputer，fit 后填充训练段数值列
# 提示：SimpleImputer(strategy=..., add_indicator=True)
imp = ____
Xi = ____
print("median statistics_:", np.round(imp.statistics_, 4).tolist())
print("填充后缺失:", int(np.isnan(Xi).sum()), "| 形状:", Xi.shape)

# TODO(5)：给上面这个需求加上缺失指示列：add_indicator=True，存入 Xi_ind
# 提示：指示列在输出矩阵的第 5 列之后，Xi_ind[:, 5:].sum() 就是缺失次数
imp_ind = ____
Xi_ind = ____
print("add_indicator 形状:", Xi_ind.shape, "= 5 原列 + 2 缺失指示列")
print("指示列合计（缺失次数）:", int(Xi_ind[:, 5:].sum()), "= 36 + 36")

# TODO(6)：类别列用 most_frequent 策略填充，存入 imp_cat
# 提示：SimpleImputer(strategy="most_frequent").fit(Xs_tr[CAT])
imp_cat = ____
print("most_frequent statistics_:", imp_cat.statistics_.tolist())
imp_const = SimpleImputer(strategy="constant", fill_value=-1).fit(Xs_tr[NUM])
print("constant(-1) 变换后最小值:", float(imp_const.transform(Xs_tr[NUM]).min()))

assert np.allclose(np.round(imp.statistics_, 4), [427.33, 35.1, 64.9, 16.0, 44.0])
assert int(np.isnan(Xi).sum()) == 0
assert Xi_ind.shape == (480, 7) and int(Xi_ind[:, 5:].sum()) == 72
assert imp_cat.statistics_.tolist() == ["变压器", "渗漏油"]
assert float(imp_const.transform(Xs_tr[NUM]).min()) == -1.0

## 题 4：OneHotEncoder 维度与基准类（对应讲解 §3.5）

In [ ]:
# TODO(7)：定义非稀疏输出的 OneHotEncoder，fit 训练段类别列并 transform
# 提示：OneHotEncoder(sparse_output=False)
oh = ____
X_oh = ____
print("设备类型类别(码位序):", oh.categories_[0].tolist())
print("缺陷类型类别(码位序):", oh.categories_[1].tolist())
print("独热维度:", X_oh.shape, "= (5 设备 + 6 缺陷) 类")

# TODO(8)：定义 drop="first" 的 OneHotEncoder 并 transform，看维度少了几列
# 提示：OneHotEncoder(sparse_output=False, drop="first")
oh_d = ____
oh_d_out = ____
print("drop=first 维度:", oh_d.transform(Xs_tr[CAT]).shape, "= 11 - 2")
print("互感器行 drop 后独热和:", float(oh_d_out[0].sum()), "← 设备类型段全 0 = 基准类")

assert oh.categories_[0].tolist() == ["互感器", "变压器", "断路器", "绝缘子", "避雷器"]
assert X_oh.shape == (480, 11)
assert oh_d.transform(Xs_tr[CAT]).shape == (480, 9)
assert float(oh_d_out[0][:5].sum()) == 0.0, "设备类型前 5 列全 0"

## 题 5：min_frequency 的维度陷阱（对应讲解 §3.6）

In [ ]:
# 训练段缺陷类型计数: 渗漏油 94 / 放电痕迹 85 / 锈蚀 84 / 发热 80 / 破损 77 / 异物 60

# TODO(9)：定义 min_frequency=0.15 的 OneHotEncoder 并 transform，观察异物 60 是否被合并
# 提示：OneHotEncoder(sparse_output=False, min_frequency=...)
oh_m1 = ____
out_m1 = ____
print("min_frequency=0.15(72): 维度", out_m1.shape, "| 只有异物 60 被合并")
print("  特征名含 infrequent:", [n for n in oh_m1.get_feature_names_out() if "infrequent" in n])

# TODO(10)：阈值抬到 0.2 再试一次：注意缺陷类型整列的遭遇
# 提示：0.2 * 480 = 96，比训练段最大的类别计数 94 还大
oh_m2 = ____
out_m2 = ____
print("min_frequency=0.2(96): 维度", out_m2.shape, "| 缺陷类型整列失效")
print("  infrequent 桶:", [c.tolist() for c in oh_m2.infrequent_categories_])

assert out_m1.shape == (480, 11), "只合并 1 个类别时维度不变"
assert "缺陷类型_infrequent_sklearn" in oh_m1.get_feature_names_out().tolist()
assert out_m2.shape == (480, 5), "整列被吞: (2 设备保留 + 1) + (0 + 1)"

**为什么这样做**：0.15 和 0.2 只差 0.05，结果从「维度不变」跳到
「整列失效」。低频治理没有免费的默认值，改一次查一次 shape。

## 题 6：未知类别的两种结局（对应讲解 §3.7）

In [ ]:
# TODO(11)：定义 handle_unknown="ignore" 的 OneHotEncoder，对含未知类别「杆塔」的行 transform
# 提示：unk 是一行 DataFrame；看输出的形状与该行独热和
oh_ign = ____
unk = ____
out_unk = ____
print("ignore 遇未知:", out_unk.shape, "| 该行独热和:", float(out_unk.sum()))

# TODO(12)：用 probe 验证：drop="first" 与 handle_unknown="ignore" 组合，fit 与 transform 是否报错
# 提示：probe(oh_bad.fit, Xs_tr[CAT]) 先接住 fit，再正常调用 transform
oh_bad = ____
st_fit, _ = ____
out_bad = ____
print("drop+ignore: fit →", st_fit, "| transform 未知 →", out_bad.shape,
      "| 独热和:", float(out_bad.sum()))

assert out_unk.shape == (1, 11)
assert float(out_unk[0][:5].sum()) == 0.0 and float(out_unk.sum()) == 1.0, "未知段归零，已知段保留"
assert st_fit == "ok" and out_bad.shape == (1, 9) and float(out_bad.sum()) == 0.0, "未知段归零"

## 题 7：OrdinalEncoder 与未知类别（对应讲解 §3.7）

In [ ]:
# TODO(13)：定义 OrdinalEncoder，fit 训练段类别列，打印码位序与前 3 行编码
# 提示：OrdinalEncoder().fit(Xs_tr[CAT])；.transform(Xs_tr[CAT].head(3))
oe = ____
oe_head = ____
print("设备类型码位序:", oe.categories_[0].tolist())
print("前 3 行编码:", oe_head.tolist())

# TODO(14)：用 probe 验证：默认 OrdinalEncoder 遇到未知类别「杆塔」会怎样
# 提示：probe(oe.transform, 单行 DataFrame)
status, msg = ____
print("OrdinalEncoder 遇未知 →", status, "|", msg)

assert oe.categories_[0].tolist() == ["互感器", "变压器", "断路器", "绝缘子", "避雷器"]
assert oe.transform(Xs_tr[CAT].head(1)).tolist()[0][0] == 3.0, "互感器 = 码位 3"
assert status == "err" and msg == "ValueError"

## 题 8：PolynomialFeatures 三连坑（对应讲解 §3.8）

In [ ]:
# TODO(15)：用 probe 验证：PolynomialFeatures 直接 fit 含 NaN 的数值列会怎样
# 提示：probe(PolynomialFeatures(degree=2).fit, Xs_tr[NUM])
status, msg = ____
print("直接对含 NaN 的数值列 fit →", status, "|", msg)

# TODO(16)：先用题 3 的 imp 填充，再定义 degree=2 / include_bias=False 的 PolynomialFeatures
# 提示：imp.transform(Xs_tr[NUM]) 得到 Xi；PolynomialFeatures(degree=2, include_bias=False).fit(Xi)
Xi = ____
pf2b = ____
Xi2 = ____
print("degree=2 无 bias:", Xi2.shape, "= 5 + 5*6/2")
pf2i = PolynomialFeatures(degree=2, include_bias=False,
                          interaction_only=True).fit(Xi)
print("interaction_only:", pf2i.transform(Xi).shape, "= 5 + 5*4/2")
pf3b = PolynomialFeatures(degree=3, include_bias=False).fit(Xi)
print("degree=3 无 bias:", pf3b.transform(Xi).shape)
print("特征名前 7:", pf2b.get_feature_names_out()[:7].tolist())
print("平方项:", [n for n in pf2b.get_feature_names_out() if "^2" in n])

# TODO(17)：找出未缩放直接 poly 的最大值来自哪个特征，并对 poly 输出再做一次 StandardScaler
# 提示：np.argmax 用 np.unravel_index 解包；StandardScaler().fit(Xi2)
mx = ____
ss2 = ____
print("未缩放直接 poly 的最大值:", round(float(Xi2.max()), 1),
      "| 来自特征:", pf2b.get_feature_names_out()[mx])
print("poly 后再 scale：x0 std", round(float(ss2.scale_[0]), 2),
      "| x0*x1 std", round(float(ss2.scale_[5]), 2), "← 交互项仍然巨大")

print("测试段 poly 形状:", pf2b.transform(imp.transform(Xs_te[NUM])).shape)

assert status == "err" and msg == "ValueError", "PolynomialFeatures 拒绝 NaN"
assert Xi2.shape == (480, 20) and pf2i.transform(Xi).shape == (480, 15)
assert pf3b.transform(Xi).shape == (480, 55)
assert round(float(Xi2.max()), 1) == 697425.4
assert pf2b.get_feature_names_out()[mx] == "x0^2"
assert pf2b.transform(imp.transform(Xs_te[NUM])).shape == (120, 20)

**为什么这样做**：先 probe 确认「poly 拒绝 NaN」再补缺失，这个顺序
本身就是考点；爆炸值 697425.4 来自 x0²，它解释了为什么升维三步曲
impute → poly → scale 少一步都不行。

---

## 综合自查

1. 三兄弟各记什么统计量？哪个和 `SimpleImputer(median)` 互证？
2. 题 2 的 72 个 NaN 是怎么来的？下游模型会遇到什么问题？
3. 题 5 两档阈值的结果差异说明什么？怎么避免「整列失效」？
4. 题 6 的组合参数在旧教程里说必报错，实测为什么能跑？风险在哪？
5. 升维三步曲是哪三步？测试段为什么要复用训练段的转换器？

全部答得上来，进入 ch03（特征选择与降维）。